# LeetCode 385 — Mini Parser bằng Python

Link đề: https://leetcode.com/problems/mini-parser/ · Độ khó: **Medium** · Chủ đề: String, Stack, Depth-First Search, Recursion, Parsing

Skeleton LeetCode:

```python
# """
# This is the interface that allows for creating nested lists.
# You should not implement it, or speculate about its implementation
# """
# class NestedInteger:
#    def __init__(self, value=None):
#        """
#        If value is not specified, initializes an empty list.
#        Otherwise initializes a single integer equal to value.
#        """
#
#    def isInteger(self):
#        """
#        @return True if this NestedInteger holds a single integer, rather than a nested list.
#        :rtype bool
#        """
#
#    def add(self, elem):
#        """
#        Set this NestedInteger to hold a nested list and adds a nested integer elem to it.
#        :rtype void
#        """
#
#    def setInteger(self, value):
#        """
#        Set this NestedInteger to hold a single integer equal to value.
#        :rtype void
#        """
#
#    def getInteger(self):
#        """
#        @return the single integer that this NestedInteger holds, if it holds a single integer
#        Return None if this NestedInteger holds a nested list
#        :rtype int
#        """
#
#    def getList(self):
#        """
#        @return the nested list that this NestedInteger holds, if it holds a nested list
#        Return None if this NestedInteger holds a single integer
#        :rtype list[NestedInteger]
#        """
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        ...
```

Ghi chú: trên LeetCode lớp `NestedInteger` **được cung cấp sẵn**; trong notebook ta **tự dựng** một bản có cùng giao diện để chạy thử. Cách `NestedInteger` lưu bên trong là do ta tự chọn, nên nếu cần hãy đối chiếu lại giao diện với đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuyển một chuỗi như `"[123,[456,[789]]]"` thành đối tượng `NestedInteger` — hoặc **một số nguyên**, hoặc **một danh sách** mà mỗi phần tử lại là `NestedInteger`.
2. Nắm các công cụ Python: duyệt chuỗi bằng chỉ số, `str.isdigit`, cắt chuỗi `s[i:j]`, `int("-12")`, **ngăn xếp** bằng `list` (`append`/`pop`), đệ quy, giới hạn đệ quy `sys.getrecursionlimit()`, `json.loads` để làm "trọng tài".
3. Đi từ **`json.loads` rồi đổi kiểu** → **đệ quy cắt chuỗi theo dấu phẩy ở mức ngoài cùng** → **đệ quy xuống bằng con trỏ** → **ngăn xếp lặp (không đệ quy)**.
4. Hiểu mấu chốt: dấu `[` là **mở một danh sách mới**, `]` là **đóng và gắn nó vào danh sách cha** — đúng hành vi của một **stack**; số được đọc **trọn cả cụm chữ số (kèm dấu `-`)**.
5. Tránh các lỗi kinh điển: đọc từng ký tự thay vì cả số, quên gắn danh sách con vào cha, quên trường hợp số đứng một mình (`"324"`), và đệ quy quá sâu (`s` dài tới `5·10^4` có thể lồng ~`25 000` tầng).

## 2. Đề bài & ví dụ

Cho chuỗi `s` là **bản tuần tự hóa (serialization)** của một danh sách lồng nhau. Hãy cài hàm `deserialize(s)` trả về đối tượng `NestedInteger` tương ứng. Mỗi phần tử là một số nguyên hoặc một danh sách mà các phần tử của nó cũng có thể là số nguyên hoặc danh sách khác.

**Ví dụ 1:** `s = "324"` → một `NestedInteger` chứa duy nhất số `324`.

**Ví dụ 2:** `s = "[123,[456,[789]]]"` → danh sách gồm 2 phần tử:
1. số nguyên `123`;
2. danh sách 2 phần tử: số `456` và danh sách chứa một số `789`.

**Ràng buộc:** `1 <= s.length <= 5·10^4`; `s` chỉ gồm chữ số, `[`, `]`, dấu `-`, và dấu `,`; `s` là tuần tự hóa **hợp lệ**; mọi giá trị trong `[-10^6, 10^6]`.

LeetCode kiểm tra bằng cách duyệt lại đối tượng bạn trả về và in ra, nên ta cũng cần hàm in để kiểm tra.

In [1]:
import json
import random
import sys
import time
from typing import List


class NestedInteger:
    """Bản tự dựng của giao diện LeetCode (trên LeetCode lớp này được cung cấp sẵn)."""

    def __init__(self, value=None):
        if value is None:                 # không có value -> danh sách rỗng
            self._int, self._list = None, []
        else:                             # có value -> một số nguyên
            self._int, self._list = value, None

    def isInteger(self):
        return self._int is not None

    def add(self, elem):
        """Biến thành danh sách (nếu cần) và thêm elem vào cuối."""
        if self._list is None:
            self._int, self._list = None, []
        self._list.append(elem)

    def setInteger(self, value):
        self._int, self._list = value, None

    def getInteger(self):
        return self._int

    def getList(self):
        return self._list

    def __repr__(self):
        return ni_to_str(self)


def ni_to_str(root):
    """Tuần tự hóa NestedInteger thành chuỗi, bằng ngăn xếp (không đệ quy -> chịu được lồng rất sâu)."""
    if root.isInteger():
        return str(root.getInteger())
    out = ["["]
    stack = [iter(root.getList())]
    first = [True]
    while stack:
        try:
            child = next(stack[-1])
        except StopIteration:
            stack.pop()
            first.pop()
            out.append("]")
            continue
        if not first[-1]:
            out.append(",")
        first[-1] = False
        if child.isInteger():
            out.append(str(child.getInteger()))
        else:
            out.append("[")
            stack.append(iter(child.getList()))
            first.append(True)
    return "".join(out)


def ni_to_py(ni):
    """Đổi NestedInteger -> int / list Python (đệ quy, chỉ dùng cho dữ liệu nông)."""
    if ni.isInteger():
        return ni.getInteger()
    return [ni_to_py(c) for c in ni.getList()]


a = NestedInteger()
a.add(NestedInteger(123))
b = NestedInteger()
b.add(NestedInteger(456))
a.add(b)
print(a, ni_to_py(a), NestedInteger(324), NestedInteger())

[123,[456]] [123, [456]] 324 []


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi bằng chỉ số, cắt chuỗi, `isdigit`, `int`

`s[i]` là ký tự thứ `i`; `s[i:j]` là đoạn từ `i` đến `j-1`; `c.isdigit()` kiểm tra một ký tự có phải chữ số; `int("-12")` đổi chuỗi (có thể kèm dấu `-`) thành số.

In [2]:
s = "[-123,45]"
print(s[0], s[1], s[1:5], s[6:8])         # [ - -123 45
print("7".isdigit(), "-".isdigit(), ",".isdigit())   # True False False
print(int("-123") + int("45"))            # -78

# đọc "một cụm số" bắt đầu ở i: đi tiếp khi còn là chữ số
i = 1
j = i + 1                                  # luôn nuốt ký tự đầu (có thể là '-')
while j < len(s) and s[j].isdigit():
    j += 1
print(s[i:j], int(s[i:j]))                # -123 -123

[ - -123 45
True False False
-78
-123 -123


### 3.2. Ngăn xếp (stack) bằng `list`

Ngăn xếp là cấu trúc **vào sau, ra trước (LIFO)**: `stack.append(x)` đẩy vào, `stack.pop()` lấy ra phần tử **mới nhất**, `stack[-1]` xem phần tử trên cùng. Rất hợp với ngoặc lồng nhau: ngoặc **mở gần nhất** sẽ được **đóng trước**.

In [3]:
stack = []
for ch in "([])":
    if ch in "([":
        stack.append(ch)
        print("mở ", ch, "->", stack)
    else:
        top = stack.pop()
        print("đóng", ch, "lấy ra", top, "->", stack)

mở  ( -> ['(']
mở  [ -> ['(', '[']
đóng ] lấy ra [ -> ['(']
đóng ) lấy ra ( -> []


### 3.3. Đệ quy và giới hạn độ sâu

Hàm gọi lại chính nó khi gặp một danh sách lồng. Python giới hạn độ sâu đệ quy mặc định ~ `1000`. Chuỗi `"[" * 5000 + "]" * 5000` lồng **5000 tầng** ⇒ hàm đệ quy theo độ sâu sẽ báo `RecursionError`.

In [4]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def depth_recursive(s, i=0):
    """Đếm độ sâu bằng đệ quy: mỗi '[' đi sâu thêm một tầng."""
    if s[i] != "[":
        return 0
    return 1 + depth_recursive(s, i + 1)

print(depth_recursive("[[[1]]]"))
try:
    depth_recursive("[" * 5000 + "1" + "]" * 5000)
except RecursionError:
    print("RecursionError: lồng 5000 tầng làm tràn ngăn xếp gọi hàm")

giới hạn đệ quy mặc định: 1000
3
RecursionError: lồng 5000 tầng làm tràn ngăn xếp gọi hàm


### 3.4. `json.loads` và vai trò "trọng tài"

Chuỗi trong đề **trùng định dạng JSON** (số, `[]`, dấu phẩy), nên `json.loads("[1,[2,3]]")` cho ngay `[1, [2, 3]]` của Python. Ta không dùng nó để "nộp" bài (đề muốn bạn tự phân tích), nhưng rất tiện làm **đáp án chuẩn độc lập** khi test.

In [5]:
print(json.loads("[123,[456,[789]]]"))
print(json.loads("324"), json.loads("[]"), json.loads("[-1,[[],-2]]"))

[123, [456, [789]]]
324 [] [-1, [[], -2]]


### 3.5. Dùng `NestedInteger` (giao diện của đề)

- `NestedInteger()` → danh sách **rỗng**; `NestedInteger(5)` → **số nguyên** `5`.
- `.add(x)` → thêm `x` vào danh sách; `.isInteger()` → là số hay danh sách; `.getInteger()`, `.getList()` → đọc giá trị.

Chú ý phân biệt: `NestedInteger()` là **danh sách rỗng**, còn `NestedInteger(0)` là **số 0** — khác hẳn nhau.

In [6]:
x = NestedInteger()          # []
x.add(NestedInteger(1))
y = NestedInteger()          # []
y.add(NestedInteger(2))
y.add(NestedInteger(3))
x.add(y)
print(x)                                       # [1,[2,3]]
print(NestedInteger(0), NestedInteger())       # 0 []
print(NestedInteger(0).isInteger(), NestedInteger().isInteger())   # True False

[1,[2,3]]
0 []
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Đọc `"[1,[2,3]]"` từ trái sang phải, như bóc **búp bê Nga**:

- Thấy `[`: "mở một hộp mới". Hộp mới nằm **bên trong** hộp đang mở (nếu có).
- Thấy một số: "bỏ số đó vào hộp đang mở gần nhất".
- Thấy `,`: chỉ là dấu ngăn cách, bỏ qua.
- Thấy `]`: "đóng hộp gần nhất" — nếu còn hộp bên ngoài thì **bỏ hộp vừa đóng vào hộp ngoài**; nếu không còn hộp nào thì đây là **kết quả cuối**.

Tính tay với `"[1,[2,3]]"` (ghi các hộp đang mở):

| Ký tự | Hành động | Các hộp đang mở (ngoài → trong) |
|---|---|---|
| `[` | mở hộp A | A=`[]` |
| `1` | bỏ 1 vào A | A=`[1]` |
| `,` | bỏ qua | A=`[1]` |
| `[` | mở hộp B | A=`[1]`, B=`[]` |
| `2` | bỏ 2 vào B | A=`[1]`, B=`[2]` |
| `,` | bỏ qua | |
| `3` | bỏ 3 vào B | A=`[1]`, B=`[2,3]` |
| `]` | đóng B, bỏ vào A | A=`[1,[2,3]]` |
| `]` | đóng A, hết hộp ngoài ⇒ kết quả | `[1,[2,3]]` |

Các hộp "đang mở" chính là một **ngăn xếp**.

## 5. Cách 1 — Brute Force / baseline: `json.loads` rồi đổi kiểu

**Ý tưởng:** nhờ thư viện biến chuỗi thành list/số Python, sau đó **đổi từng phần tử** thành `NestedInteger` bằng đệ quy:

- Nếu là `int` ⇒ `NestedInteger(x)`.
- Nếu là list ⇒ `NestedInteger()` rồi `add` từng phần tử đã đổi.

Đây là "gian lận hợp lệ" để có baseline và đáp án đối chiếu, chứ không phải điều đề muốn (phỏng vấn sẽ yêu cầu tự phân tích chuỗi).

In [7]:
def from_py(obj):
    if isinstance(obj, int):
        return NestedInteger(obj)
    ni = NestedInteger()
    for x in obj:
        ni.add(from_py(x))
    return ni


class SolutionJson:
    """Cách 1: json.loads + đổi kiểu đệ quy."""

    def deserialize(self, s: str) -> NestedInteger:
        return from_py(json.loads(s))


for s in ["324", "[123,[456,[789]]]", "[]", "[[],-1]"]:
    r = SolutionJson().deserialize(s)
    print(f"{s!r:<22} ->", r)
    assert ni_to_str(r) == s

'324'                  -> 324
'[123,[456,[789]]]'    -> [123,[456,[789]]]
'[]'                   -> []
'[[],-1]'              -> [[],-1]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ; nhưng `json.loads` và `from_py` đều **đệ quy** theo độ sâu nên hỏng ở độ sâu lớn.

**Nút thắt:** (1) dựa vào thư viện nên không học được gì về phân tích chuỗi; (2) chết khi lồng sâu (~1000+ tầng với `from_py`). Ta tự phân tích chuỗi, và về lâu dài muốn **không đệ quy**.

## 6. Ý tưởng thuật toán / bước đột phá

Cấu trúc của chuỗi là **đệ quy**: một "phần tử" hoặc là **số**, hoặc là `[` + (các phần tử cách nhau bởi `,`) + `]`.

Ba cách tự nhiên để khai thác:

- **(a) Chia theo dấu phẩy ở mức ngoài cùng:** bỏ cặp `[ ]` ngoài, đếm độ sâu ngoặc để tìm các dấu `,` **ở độ sâu 0**; mỗi đoạn giữa hai dấu phẩy là một phần tử con ⇒ gọi đệ quy trên đoạn đó (cắt chuỗi).
- **(b) Đệ quy xuống kèm con trỏ:** một con trỏ `i` chạy qua chuỗi; gặp `[` thì tạo danh sách và **lặp đọc phần tử** cho tới `]`, mỗi phần tử gọi lại chính hàm.
- **(c) Ngăn xếp tường minh:** bỏ đệ quy; dùng `stack` các `NestedInteger` đang mở (đúng bảng ở mục 4).

(b) và (c) chỉ đi qua chuỗi **một lần** (`O(n)`). (c) còn không bị giới hạn độ sâu đệ quy của Python.

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (`json` + đổi kiểu, mục 5), Cách 2 (đệ quy cắt chuỗi theo dấu phẩy mức ngoài), Cách 3 (đệ quy với con trỏ), Cách 4 (ngăn xếp lặp — **chính thức**).

### Cách 2 — Đệ quy: chia theo dấu phẩy ở độ sâu 0

Giải thích từng đoạn:

1. Nếu `s == "[]"` ⇒ trả danh sách rỗng `NestedInteger()`.
2. Nếu `s[0] != '['` ⇒ là số ⇒ `NestedInteger(int(s))`.
3. Ngược lại là danh sách: `ans = NestedInteger()`; biến `depth = 0` đếm độ sâu ngoặc (không tính cặp ngoài cùng), `j = 1` là đầu đoạn hiện tại.
4. Duyệt `i` từ `1`:
   - Nếu `depth == 0` và (`s[i] == ','` hoặc `i` là ký tự cuối): đoạn `s[j:i]` là **một phần tử con** ⇒ `ans.add(deserialize(s[j:i]))`, đặt `j = i + 1`.
   - Ngược lại `[` ⇒ `depth += 1`, `]` ⇒ `depth -= 1`.

Chú ý ký tự cuối là `]` đóng ngoài cùng nên `s[j:i]` không chứa nó.

In [8]:
class SolutionSplit:
    """Cách 2: đệ quy cắt chuỗi theo dấu phẩy ở độ sâu 0."""

    def deserialize(self, s: str) -> NestedInteger:
        if s == "[]":
            return NestedInteger()
        if s[0] != "[":
            return NestedInteger(int(s))
        ans = NestedInteger()
        depth, j = 0, 1
        for i in range(1, len(s)):
            if depth == 0 and (s[i] == "," or i == len(s) - 1):
                ans.add(self.deserialize(s[j:i]))       # một phần tử con
                j = i + 1
            elif s[i] == "[":
                depth += 1
            elif s[i] == "]":
                depth -= 1
        return ans


for s in ["324", "[123,[456,[789]]]", "[]", "[[],-1]", "[[[]]]"]:
    r = SolutionSplit().deserialize(s)
    print(f"{s!r:<22} ->", r)
    assert ni_to_str(r) == s

'324'                  -> 324
'[123,[456,[789]]]'    -> [123,[456,[789]]]
'[]'                   -> []
'[[],-1]'              -> [[],-1]
'[[[]]]'               -> [[[]]]


**Code nộp LeetCode cho Cách 2:**

```python
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        if s == "[]":
            return NestedInteger()
        if s[0] != "[":
            return NestedInteger(int(s))
        ans = NestedInteger()
        depth, j = 0, 1
        for i in range(1, len(s)):
            if depth == 0 and (s[i] == "," or i == len(s) - 1):
                ans.add(self.deserialize(s[j:i]))
                j = i + 1
            elif s[i] == "[":
                depth += 1
            elif s[i] == "]":
                depth -= 1
        return ans
```

**Độ phức tạp:** mỗi tầng quét lại đoạn của nó và **sao chép** chuỗi con ⇒ tệ nhất **`O(n · độ_sâu)`** (có thể `O(n²)` với chuỗi lồng sâu), bộ nhớ `O(n)`; đệ quy sâu ⇒ có thể `RecursionError`.

---

### Cách 3 — Đệ quy xuống với con trỏ

Giải thích từng đoạn:

1. Giữ một con trỏ `i` (chỉ số hiện tại) dùng chung cho mọi lần gọi (biến `nonlocal`).
2. `parse()`: nếu `s[i] == '['` ⇒ `i += 1`, tạo `ni = NestedInteger()`; **lặp** đến khi `s[i] == ']'`: nếu là `,` thì bỏ qua (`i += 1`), ngược lại `ni.add(parse())`. Cuối cùng `i += 1` (nuốt `]`) và trả `ni`.
3. Ngược lại là **số**: đọc trọn cụm `-?digits` (xem mục 3.1), `i` nhảy tới sau cụm đó, trả `NestedInteger(int(...))`.

Mỗi ký tự được đọc **đúng một lần** ⇒ `O(n)`; không cắt chuỗi.

In [9]:
class SolutionPointer:
    """Cách 3: đệ quy với con trỏ i dùng chung."""

    def deserialize(self, s: str) -> NestedInteger:
        n = len(s)
        i = 0

        def parse() -> NestedInteger:
            nonlocal i
            if s[i] == "[":
                i += 1                                   # nuốt '['
                ni = NestedInteger()
                while s[i] != "]":
                    if s[i] == ",":
                        i += 1
                    else:
                        ni.add(parse())
                i += 1                                   # nuốt ']'
                return ni
            j = i + 1                                    # số: nuốt cả '-' nếu có
            while j < n and s[j].isdigit():
                j += 1
            ni = NestedInteger(int(s[i:j]))
            i = j
            return ni

        return parse()


for s in ["324", "[123,[456,[789]]]", "[]", "[[],-1]", "[-1,[-22,[333]],0]"]:
    r = SolutionPointer().deserialize(s)
    print(f"{s!r:<22} ->", r)
    assert ni_to_str(r) == s

'324'                  -> 324
'[123,[456,[789]]]'    -> [123,[456,[789]]]
'[]'                   -> []
'[[],-1]'              -> [[],-1]
'[-1,[-22,[333]],0]'   -> [-1,[-22,[333]],0]


**Code nộp LeetCode cho Cách 3:**

```python
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        n = len(s)
        i = 0

        def parse() -> NestedInteger:
            nonlocal i
            if s[i] == "[":
                i += 1
                ni = NestedInteger()
                while s[i] != "]":
                    if s[i] == ",":
                        i += 1
                    else:
                        ni.add(parse())
                i += 1
                return ni
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            ni = NestedInteger(int(s[i:j]))
            i = j
            return ni

        return parse()
```

**Độ phức tạp:** `O(n)` thời gian; bộ nhớ `O(độ_sâu)` cho ngăn xếp gọi hàm (cộng `O(n)` cho kết quả). **Rủi ro:** độ sâu tới `~25 000` làm `RecursionError` trong Python.

---

### Cách 4 — Ngăn xếp lặp (không đệ quy) — chính thức

Đây chính là bảng ở mục 4. Giải thích từng đoạn:

1. Nếu `s[0] != '['` ⇒ cả chuỗi là **một số** ⇒ trả `NestedInteger(int(s))` (xử lý riêng, vì ngăn xếp sẽ rỗng).
2. `stack = []` chứa các danh sách **đang mở**; `ans = None`.
3. Duyệt `i`:
   - `[`: `stack.append(NestedInteger())`.
   - `]`: `done = stack.pop()`; nếu `stack` còn ⇒ `stack[-1].add(done)`; ngược lại `ans = done` (xong).
   - `,`: bỏ qua.
   - còn lại (`-` hoặc chữ số): đọc trọn cụm số `s[i:j]`, `stack[-1].add(NestedInteger(int(...)))`, nhảy `i = j`.
4. Trả `ans`.

In [10]:
class SolutionStack:
    """Cách 4 (demo): ngăn xếp các NestedInteger đang mở."""

    def deserialize(self, s: str) -> NestedInteger:
        if s[0] != "[":
            return NestedInteger(int(s))
        stack = []
        ans = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "[":
                stack.append(NestedInteger())            # mở hộp mới
                i += 1
            elif c == "]":
                done = stack.pop()                       # đóng hộp trên cùng
                if stack:
                    stack[-1].add(done)                  # gắn vào hộp cha
                else:
                    ans = done                           # hết hộp ngoài: kết quả
                i += 1
            elif c == ",":
                i += 1
            else:                                        # số: '-' hoặc chữ số
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                stack[-1].add(NestedInteger(int(s[i:j])))
                i = j
        return ans


for s in ["324", "[123,[456,[789]]]", "[]", "[[],-1]", "[-1,[-22,[333]],0]"]:
    r = SolutionStack().deserialize(s)
    print(f"{s!r:<22} ->", r)
    assert ni_to_str(r) == s

'324'                  -> 324
'[123,[456,[789]]]'    -> [123,[456,[789]]]
'[]'                   -> []
'[[],-1]'              -> [[],-1]
'[-1,[-22,[333]],0]'   -> [-1,[-22,[333]],0]


**Code nộp LeetCode cho Cách 4:**

```python
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        if s[0] != "[":
            return NestedInteger(int(s))
        stack = []
        ans = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "[":
                stack.append(NestedInteger())
                i += 1
            elif c == "]":
                done = stack.pop()
                if stack:
                    stack[-1].add(done)
                else:
                    ans = done
                i += 1
            elif c == ",":
                i += 1
            else:
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                stack[-1].add(NestedInteger(int(s[i:j])))
                i = j
        return ans
```

**Độ phức tạp:** `O(n)` thời gian (mỗi ký tự đi qua một lần), bộ nhớ `O(độ_sâu)` cho ngăn xếp (cộng `O(n)` cho kết quả). **Không giới hạn độ sâu**.

## 8. TRACE từng bước

Trace Cách 4 trên `s = "[123,[456,[789]]]"`. Mỗi dòng: chỉ số `i`, token đọc được, hành động và **ngăn xếp** (mỗi hộp in dạng chuỗi, ngoài → trong).

| `i` | Token | Hành động | Ngăn xếp |
|---|---|---|---|
| 0 | `[` | mở hộp | `[]` |
| 1 | `123` | thêm số vào hộp trên cùng | `[123]` |
| 4 | `,` | bỏ qua | `[123]` |
| 5 | `[` | mở hộp | `[123]`, `[]` |
| 6 | `456` | thêm số | `[123]`, `[456]` |
| 9 | `,` | bỏ qua | |
| 10 | `[` | mở hộp | `[123]`, `[456]`, `[]` |
| 11 | `789` | thêm số | ..., `[789]` |
| 14 | `]` | đóng `[789]`, gắn vào `[456]` | `[123]`, `[456,[789]]` |
| 15 | `]` | đóng, gắn vào `[123]` | `[123,[456,[789]]]` |
| 16 | `]` | đóng; hết hộp cha ⇒ kết quả | (rỗng) ⇒ ans |

Hàm dưới in đúng từng bước như vậy cho một chuỗi bất kỳ.

In [11]:
def trace_stack(s):
    print(f"s = {s!r}")
    if s[0] != "[":
        print("  là một số đứng một mình ->", int(s))
        return NestedInteger(int(s))
    print(f"{'i':<4}{'token':<8}{'hành động':<28}ngăn xếp")
    stack, ans = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "[":
            stack.append(NestedInteger())
            act, tok, nxt = "mở hộp", c, i + 1
        elif c == "]":
            done = stack.pop()
            if stack:
                stack[-1].add(done)
                act = f"đóng {done} -> gắn vào cha"
            else:
                ans = done
                act = f"đóng {done} -> KẾT QUẢ"
            tok, nxt = c, i + 1
        elif c == ",":
            act, tok, nxt = "bỏ qua", c, i + 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            stack[-1].add(NestedInteger(int(s[i:j])))
            act, tok, nxt = "thêm số vào hộp trên", s[i:j], j
        print(f"{i:<4}{tok:<8}{act:<28}{[str(x) for x in stack]}")
        i = nxt
    print("=>", ans, "\n")
    return ans


r = trace_stack("[123,[456,[789]]]")
assert ni_to_str(r) == "[123,[456,[789]]]"
trace_stack("[-1,[],[2,[-3]]]")
trace_stack("324")

s = '[123,[456,[789]]]'
i   token   hành động                   ngăn xếp
0   [       mở hộp                      ['[]']
1   123     thêm số vào hộp trên        ['[123]']
4   ,       bỏ qua                      ['[123]']
5   [       mở hộp                      ['[123]', '[]']
6   456     thêm số vào hộp trên        ['[123]', '[456]']
9   ,       bỏ qua                      ['[123]', '[456]']
10  [       mở hộp                      ['[123]', '[456]', '[]']
11  789     thêm số vào hộp trên        ['[123]', '[456]', '[789]']
14  ]       đóng [789] -> gắn vào cha   ['[123]', '[456,[789]]']
15  ]       đóng [456,[789]] -> gắn vào cha['[123,[456,[789]]]']
16  ]       đóng [123,[456,[789]]] -> KẾT QUẢ[]
=> [123,[456,[789]]] 

s = '[-1,[],[2,[-3]]]'
i   token   hành động                   ngăn xếp
0   [       mở hộp                      ['[]']
1   -1      thêm số vào hộp trên        ['[-1]']
3   ,       bỏ qua                      ['[-1]']
4   [       mở hộp                      ['[-1]', '[]']


324

Để thấy rõ sự khác biệt giữa các cách, hãy xem **hình dạng lời gọi** của Cách 2 (đệ quy cắt chuỗi) trên cùng chuỗi: nó gọi `deserialize` trên từng **chuỗi con**:

In [12]:
def trace_split(s, depth=0):
    print("  " * depth + f"deserialize({s!r})")
    if s == "[]" or s[0] != "[":
        return
    d, j = 0, 1
    for i in range(1, len(s)):
        if d == 0 and (s[i] == "," or i == len(s) - 1):
            trace_split(s[j:i], depth + 1)
            j = i + 1
        elif s[i] == "[":
            d += 1
        elif s[i] == "]":
            d -= 1


trace_split("[123,[456,[789]]]")

deserialize('[123,[456,[789]]]')
  deserialize('123')
  deserialize('[456,[789]]')
    deserialize('456')
    deserialize('[789]')
      deserialize('789')


## 9. Code nộp LeetCode

Phiên bản cuối (Cách 4 — ngăn xếp), đúng tên lớp/method/signature (`NestedInteger` do LeetCode cung cấp; ở đây dùng bản tự dựng ở mục 2):

In [13]:
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        if s[0] != "[":                       # cả chuỗi là một số
            return NestedInteger(int(s))
        stack = []
        ans = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "[":
                stack.append(NestedInteger())
                i += 1
            elif c == "]":
                done = stack.pop()
                if stack:
                    stack[-1].add(done)
                else:
                    ans = done
                i += 1
            elif c == ",":
                i += 1
            else:                              # '-' hoặc chữ số: đọc trọn cụm số
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                stack[-1].add(NestedInteger(int(s[i:j])))
                i = j
        return ans


print(Solution().deserialize("324"))
print(Solution().deserialize("[123,[456,[789]]]"))

324
[123,[456,[789]]]


## 10. Vì sao thuật toán đúng

Gọi "hộp mở" là danh sách đã bắt đầu (`[`) nhưng chưa gặp `]` tương ứng. **Bất biến:** tại mọi thời điểm, `stack` chứa đúng các hộp mở, **từ ngoài vào trong**, mỗi hộp chứa các phần tử con đã được đọc xong của nó (theo thứ tự).

- **`[`:** bắt đầu danh sách mới, nằm **trong** hộp trên cùng hiện tại (sẽ được gắn vào khi đóng) ⇒ đẩy lên đỉnh. Bất biến giữ.
- **Số:** thuộc về hộp **gần nhất chưa đóng** — chính là đỉnh ngăn xếp (vì chuỗi hợp lệ, số chỉ nằm trong ngoặc) ⇒ `stack[-1].add(...)`. Đọc **trọn cụm** `-?[0-9]+` đảm bảo giá trị đúng (nhiều chữ số, số âm). Dấu `-` chỉ xuất hiện ngay đầu số nên `j = i + 1` rồi nuốt tiếp chữ số là đủ.
- **`]`:** kết thúc hộp ở đỉnh. Hộp đó là một phần tử **hoàn chỉnh** của hộp ngay dưới nó (nếu có) ⇒ gắn vào cha ở đúng vị trí cuối (đúng thứ tự xuất hiện). Nếu không có cha, nó là toàn bộ kết quả.
- **`,`:** chỉ phân cách, không làm đổi cấu trúc.

Chuỗi hợp lệ ⇒ ngăn xếp không bao giờ pop rỗng và kết thúc đúng lúc rỗng. Mỗi ký tự xử lý một lần ⇒ `O(n)`. Trường hợp số đứng một mình (`"324"`) không có ngoặc nên xử lý riêng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space (phụ) | Độ sâu tối đa trong Python |
|---|---|---|---|
| 1. `json` + đổi kiểu | `O(n)` | `O(độ_sâu)` đệ quy | ~ vài trăm đến ~1000 tầng |
| 2. Đệ quy cắt chuỗi | `O(n · độ_sâu)` (tệ nhất `O(n²)`) | `O(n)` chuỗi con | ~1000 tầng |
| 3. Đệ quy con trỏ | `O(n)` | `O(độ_sâu)` đệ quy | ~1000 tầng |
| 4. Ngăn xếp lặp | `O(n)` | `O(độ_sâu)` ngăn xếp | **không giới hạn** (chỉ bởi bộ nhớ) |

Với `n = 5·10^4`, độ sâu lồng tối đa là `n/2 = 25 000`. Hai cách đệ quy nhanh gặp `RecursionError` (trừ khi tăng giới hạn), nên **ngăn xếp tường minh** là lựa chọn an toàn nhất trên LeetCode với Python.

(Kết quả `NestedInteger` tự nó chiếm `O(n)` bộ nhớ; "Space phụ" ở bảng là phần thêm ngoài kết quả.)

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề:** `"324"`, `"[123,[456,[789]]]"`.
2. **Edge case:** số âm, số `0`, nhiều chữ số, `[]`, `[[]]`, `[[],[]]`, số âm trong danh sách lồng, biên `±10^6`.
3. **Đối chiếu nhiều cách bằng `assert`:** sinh **cấu trúc Python ngẫu nhiên có seed** (đúng ràng buộc giá trị), tuần tự hóa bằng `json.dumps` (đáp án độc lập), rồi mọi cách phải cho ra cấu trúc giống hệt.
4. **Round-trip:** `ni_to_str(deserialize(s)) == s` (chuỗi chuẩn không có khoảng trắng).
5. **Lồng sâu 24 000 tầng & rất rộng:** chỉ cách ngăn xếp chịu được độ sâu; các cách đệ quy phải báo `RecursionError` (ta bắt và in ra).

In [14]:
ALL = [SolutionJson, SolutionSplit, SolutionPointer, SolutionStack, Solution]

# 1) Ví dụ đề
for cls in ALL:
    assert ni_to_py(cls().deserialize("324")) == 324
    assert ni_to_py(cls().deserialize("[123,[456,[789]]]")) == [123, [456, [789]]]
print("Ví dụ đề: OK")

# 2) Edge cases (cặp: chuỗi, cấu trúc Python mong đợi)
edge = [
    ("0", 0), ("-0", 0), ("7", 7), ("-1", -1), ("1000000", 10**6), ("-1000000", -10**6),
    ("[]", []), ("[[]]", [[]]), ("[[],[]]", [[], []]), ("[[[]]]", [[[]]]),
    ("[0]", [0]), ("[-1]", [-1]), ("[1,2,3]", [1, 2, 3]),
    ("[-1,[-22,[333]],0]", [-1, [-22, [333]], 0]),
    ("[[],-5,[[],[7]]]", [[], -5, [[], [7]]]),
    ("[1000000,-1000000]", [10**6, -10**6]),
]
for cls in ALL:
    for s, expected in edge:
        got = ni_to_py(cls().deserialize(s))
        assert got == expected, (cls.__name__, s, got, expected)
        assert got == json.loads(s)
print("Edge cases:", len(edge), "chuỗi x", len(ALL), "cách: OK")

# kiểm tra kiểu: NestedInteger() là danh sách rỗng, NestedInteger(0) là số 0
assert Solution().deserialize("0").isInteger() and not Solution().deserialize("[]").isInteger()
assert Solution().deserialize("[0]").getList()[0].isInteger()

Ví dụ đề: OK
Edge cases: 16 chuỗi x 5 cách: OK


In [15]:
# 3) Đối chiếu nhiều cách trên cấu trúc ngẫu nhiên (seed cố định), đúng ràng buộc giá trị [-10^6, 10^6]
def gen_py(rng, depth, max_len):
    if depth == 0 or rng.random() < 0.3:
        return rng.choice([0, rng.randint(-9, 9), rng.randint(-10**6, 10**6)])
    return [gen_py(rng, depth - 1, max_len) for _ in range(rng.randint(0, max_len))]


def to_str(obj):
    return json.dumps(obj, separators=(",", ":"))


rng = random.Random(385)
cases = 0
for _ in range(600):
    obj = gen_py(rng, rng.randint(0, 6), rng.randint(0, 5))
    s = to_str(obj)
    assert 1 <= len(s) <= 5 * 10**4
    for cls in ALL:
        ni = cls().deserialize(s)
        assert ni_to_py(ni) == obj, (cls.__name__, s)       # đúng cấu trúc
        assert ni_to_str(ni) == s, (cls.__name__, s)         # round-trip
    cases += 1
print("Ngẫu nhiên:", cases, "chuỗi, 5 cách đều khớp cấu trúc Python (json) và round-trip")

Ngẫu nhiên: 600 chuỗi, 5 cách đều khớp cấu trúc Python (json) và round-trip


In [16]:
# 3b) Số nguyên đứng một mình, nhiều giá trị ngẫu nhiên
rng = random.Random(1385)
for _ in range(300):
    v = rng.randint(-10**6, 10**6)
    for cls in ALL:
        ni = cls().deserialize(str(v))
        assert ni.isInteger() and ni.getInteger() == v
print("Số đứng một mình: OK")

# 4) Chuỗi rộng: 5000 số trong một danh sách (nông nhưng dài)
vals = [rng.randint(-10**6, 10**6) for _ in range(5000)]
wide = to_str(vals)
assert len(wide) <= 5 * 10**4
for cls in ALL:
    ni = cls().deserialize(wide)
    assert [c.getInteger() for c in ni.getList()] == vals
print("Chuỗi rộng", len(wide), "ký tự: OK cho cả 5 cách")

Số đứng một mình: OK
Chuỗi rộng 36866 ký tự: OK cho cả 5 cách


In [17]:
# 5) Lồng cực sâu: "[" * 24000 + "]" * 24000  (dài 48000 <= 5*10^4)
deep = "[" * 24000 + "]" * 24000
assert len(deep) <= 5 * 10**4
for cls in ALL:
    try:
        ni = cls().deserialize(deep)
        ok = ni_to_str(ni) == deep
        print(f"{cls.__name__:<16} lồng 24000 tầng: {'chạy được, đúng' if ok else 'SAI'}")
        assert ok
    except RecursionError:
        print(f"{cls.__name__:<16} lồng 24000 tầng: RecursionError (cách đệ quy)")

# lồng sâu có số ở giữa
deep2 = "[" * 20000 + "-123" + "]" * 20000
ni = SolutionStack().deserialize(deep2)
assert ni_to_str(ni) == deep2
print("Ngăn xếp: lồng 20000 tầng có số ở giữa: OK")

SolutionJson     lồng 24000 tầng: RecursionError (cách đệ quy)


SolutionSplit    lồng 24000 tầng: RecursionError (cách đệ quy)
SolutionPointer  lồng 24000 tầng: RecursionError (cách đệ quy)
SolutionStack    lồng 24000 tầng: chạy được, đúng
Solution         lồng 24000 tầng: chạy được, đúng
Ngăn xếp: lồng 20000 tầng có số ở giữa: OK


In [18]:
# 6) Thời gian: Cách 2 (cắt chuỗi) chậm dần với độ sâu, Cách 4 tuyến tính  (chỉ in, không assert)
sys.setrecursionlimit(20000)
for depth in (200, 800):
    s = "[" * depth + "1" + "]" * depth
    row = []
    for cls in (SolutionSplit, SolutionPointer, SolutionStack):
        t = time.perf_counter()
        cls().deserialize(s)
        row.append(f"{cls.__name__}={time.perf_counter() - t:.4f}s")
    print(f"độ sâu {depth:>4}:", "  ".join(row))
sys.setrecursionlimit(1000)

độ sâu  200: SolutionSplit=0.0018s  SolutionPointer=0.0001s  SolutionStack=0.0001s
độ sâu  800: SolutionSplit=0.0308s  SolutionPointer=0.0003s  SolutionStack=0.0002s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đọc từng ký tự thay vì trọn cả số

Xử lý mỗi ký tự riêng lẻ: `"123"` bị tách thành `1`, `2`, `3`, còn `-` gây lỗi `int("-")`.

In [19]:
def bad_digit_by_digit(s):
    if s[0] != "[":
        return NestedInteger(int(s))
    stack, ans = [], None
    for c in s:
        if c == "[":
            stack.append(NestedInteger())
        elif c == "]":
            done = stack.pop()
            if stack: stack[-1].add(done)
            else: ans = done
        elif c == ",":
            pass
        else:
            stack[-1].add(NestedInteger(int(c)))       # SAI: chỉ một chữ số, và '-' lỗi
    return ans

print(bad_digit_by_digit("[123,[456]]"), "(đề muốn [123,[456]])")
try:
    bad_digit_by_digit("[-5]")
except ValueError as e:
    print("ValueError:", e)

[1,2,3,[4,5,6]] (đề muốn [123,[456]])
ValueError: invalid literal for int() with base 10: '-'


**Sửa:** từ vị trí `i`, nuốt `j = i + 1` rồi tiếp tục khi `s[j].isdigit()`; sau đó `int(s[i:j])`.

### Lỗi 2: quên gắn danh sách con vào cha khi gặp `]`

Chỉ `pop` mà không `add` vào hộp bên dưới ⇒ phần lồng bên trong **biến mất**.

In [20]:
def bad_no_attach(s):
    if s[0] != "[":
        return NestedInteger(int(s))
    stack, ans = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "[":
            stack.append(NestedInteger()); i += 1
        elif c == "]":
            done = stack.pop()                 # SAI: thiếu stack[-1].add(done)
            ans = done
            i += 1
        elif c == ",":
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit(): j += 1
            stack[-1].add(NestedInteger(int(s[i:j]))); i = j
    return ans

print(bad_no_attach("[1,[2,3]]"), "(đề muốn [1,[2,3]])")

[1] (đề muốn [1,[2,3]])


**Sửa:** sau `pop`, nếu `stack` còn phần tử thì `stack[-1].add(done)`; nếu hết thì đó là `ans`. (Lưu ý biến `ans = done` ở đây còn bị **ghi đè** bởi hộp con đóng cuối cùng — kết quả sai nặng hơn.)

### Lỗi 3: quên trường hợp **số đứng một mình** (`"324"`)

Chạy ngay vòng lặp ngăn xếp trên `"324"`: gặp chữ số nhưng `stack` rỗng ⇒ `IndexError`.

In [21]:
def bad_no_scalar(s):
    stack, ans = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "[":
            stack.append(NestedInteger()); i += 1
        elif c == "]":
            done = stack.pop()
            if stack: stack[-1].add(done)
            else: ans = done
            i += 1
        elif c == ",":
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit(): j += 1
            stack[-1].add(NestedInteger(int(s[i:j])))       # stack rỗng khi s = "324"
            i = j
    return ans

try:
    bad_no_scalar("324")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Sửa:** đầu hàm `if s[0] != '[': return NestedInteger(int(s))`.

### Lỗi 4: đệ quy khi dữ liệu lồng quá sâu

Cách 2 hoặc 3 chạy đúng với test nhỏ nhưng `RecursionError` khi chuỗi lồng hàng nghìn tầng (đề cho phép ~25 000).

In [22]:
deep = "[" * 3000 + "]" * 3000
for cls in (SolutionPointer, SolutionStack):
    try:
        r = cls().deserialize(deep)
        print(cls.__name__, "-> chạy được")
    except RecursionError:
        print(cls.__name__, "-> RecursionError")

SolutionPointer -> RecursionError
SolutionStack -> chạy được


**Sửa:** dùng **ngăn xếp tường minh** (Cách 4), hoặc (ít gọn hơn) `sys.setrecursionlimit` + chạy trong thread có stack lớn.

### Lỗi 5: nhầm `NestedInteger()` (danh sách rỗng) với `NestedInteger(0)` (số 0)

Viết `NestedInteger(int(x)) if x else NestedInteger()` làm `0` bị đọc thành **danh sách rỗng**. Trong bản tự dựng ta kiểm tra `value is None` (không dùng `if value:`) đúng vì lý do này.

In [23]:
class BadNI:
    def __init__(self, value=None):
        self._int = value if value else None       # SAI: 0 là falsy -> thành None
        self._list = [] if not value else None

print(BadNI(0)._int, "(đúng phải là 0, không phải None)")
print(NestedInteger(0).isInteger(), "(bản đúng)")

None (đúng phải là 0, không phải None)
True (bản đúng)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Ngoặc lồng nhau / cấu trúc đệ quy trong chuỗi** (`[]`, `()`, `{}`) ⇒ nghĩ ngay **ngăn xếp**: mở ⇒ đẩy, đóng ⇒ lấy ra và **trao cho cha**.
- Python **đệ quy kém sâu**; nếu đề cho độ sâu ~`10^4` trở lên ⇒ chuyển sang **ngăn xếp tường minh**.
- **Đọc cả cụm số** (`-?\d+`) rồi mới `int(...)`; đừng xử lý từng chữ số.
- Nhớ **trường hợp biên đầu vào là một số** (không ngoặc) — rất hay bị quên.
- Có `json.loads` để làm **trọng tài** kiểm thử (khi định dạng trùng JSON). Dùng nó trong test chứ không dùng làm đáp án nộp.
- Bài này cùng họ với: Decode String (394), Basic Calculator (224), Flatten Nested List Iterator (341), Valid Parentheses (20).

## 15. Biến thể của bài

1. **LeetCode 339/364 — Nested List Weight Sum (I/II):** tính tổng mỗi số nhân với **độ sâu** (hoặc độ sâu ngược).
2. **LeetCode 341 — Flatten Nested List Iterator:** duyệt phẳng mọi số theo thứ tự (thường dùng ngăn xếp các iterator).
3. **LeetCode 394 — Decode String:** `3[a2[c]]` → `accaccacc`, cùng khung ngăn xếp.
4. **Chuỗi có khoảng trắng / số thực / chuỗi ký tự trong ngoặc kép:** cần thêm bước **tách token** (tokenizer) trước.

Cài thử biến thể 1 (tổng có trọng số độ sâu) trên đối tượng `NestedInteger`, bằng ngăn xếp:

In [24]:
def depth_sum(ni):
    """Tổng mỗi số nhân độ sâu (số ở danh sách ngoài cùng có độ sâu 1). Lặp, không đệ quy."""
    if ni.isInteger():
        return ni.getInteger()          # coi như độ sâu 1
    total = 0
    stack = [(ni, 0)]                   # (danh sách, độ sâu của danh sách)
    while stack:
        node, d = stack.pop()
        for child in node.getList():
            if child.isInteger():
                total += child.getInteger() * (d + 1)
            else:
                stack.append((child, d + 1))
    return total


# [1,[4,[6]]] -> 1*1 + 4*2 + 6*3 = 27
assert depth_sum(Solution().deserialize("[1,[4,[6]]]")) == 27
# [[1,1],2,[1,1]] -> 4*2 + 2*1 = 10
assert depth_sum(Solution().deserialize("[[1,1],2,[1,1]]")) == 10
print("depth_sum OK")

# Giải mã chuỗi lặp: 3[a2[c]] -> accaccacc (cùng tư tưởng ngăn xếp)
def decode_string(s):
    stack, cur, num = [], "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, num)); cur, num = "", 0
        elif ch == "]":
            prev, k = stack.pop(); cur = prev + cur * k
        else:
            cur += ch
    return cur

print(decode_string("3[a2[c]]"))
assert decode_string("3[a2[c]]") == "accaccacc"

depth_sum OK
accaccacc


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (phụ) | Khi nào dùng |
|---|---|---|---|---|
| 1. `json` + đổi kiểu | thư viện parse, rồi bọc lại | `O(n)` | `O(độ_sâu)` | Làm trọng tài khi test; không nộp phỏng vấn |
| 2. Đệ quy cắt chuỗi | chia theo `,` ở độ sâu 0, đệ quy trên chuỗi con | `O(n·độ_sâu)` | `O(n)` | Gọn, dễ nghĩ; input nhỏ/nông |
| 3. Đệ quy con trỏ | một con trỏ chạy qua chuỗi, đệ quy theo phần tử | `O(n)` | `O(độ_sâu)` | Tự nhiên với "ngữ pháp" đệ quy; độ sâu vừa phải |
| 4. Ngăn xếp lặp | đẩy khi `[`, gắn vào cha khi `]` | `O(n)` | `O(độ_sâu)` | **Nộp LeetCode với Python**, an toàn với lồng sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. Với `s = "[-12,[3]]"`, (a) `s[1:4]` là gì? (b) `int(s[1:4])` là gì? (c) `"-".isdigit()`? (d) `NestedInteger(0).isInteger()`?

**Bài 2 (tính tay).** Với `s = "[1,[],[2,[3]]]"` lập bảng các bước của Cách 4 (ký tự/token, hành động, ngăn xếp), và viết kết quả cuối bằng ký hiệu `[...]`.

**Bài 3 (điền TODO).** Hoàn thành `parse_stack_todo(s)` theo Cách 4 (trả `NestedInteger`).

In [25]:
def parse_stack_todo(s):
    # TODO 1: nếu s[0] != '[' thì trả NestedInteger(int(s))
    # TODO 2: stack = [], ans = None, duyệt i qua s
    # TODO 3: '[' -> đẩy NestedInteger() ; ',' -> bỏ qua
    # TODO 4: ']' -> pop; nếu stack còn thì stack[-1].add(done) ngược lại ans = done
    # TODO 5: số -> đọc trọn cụm (nuốt '-' ở đầu), stack[-1].add(NestedInteger(int(...)))
    return None

# KHÔNG gọi parse_stack_todo ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `class SolutionRecursive2` theo Cách 3 (đệ quy với con trỏ) **mà không nhìn lại** mục 7; kiểm tra bằng 100 chuỗi ngẫu nhiên có seed và so với `json.loads`.

**Bài 5 (biến thể / mini challenge).** Viết `max_depth(s)` trả **độ sâu lồng lớn nhất** của chuỗi `s` (`"324"` → `0`, `"[]"` → `1`, `"[1,[2,[3]]]"` → `3`) **mà không** tạo `NestedInteger` và **không** đệ quy. Thêm `flatten(s)` trả list phẳng các số theo thứ tự xuất hiện (`"[1,[2,[3]],4]"` → `[1,2,3,4]`) bằng cách duyệt đối tượng `NestedInteger` đã parse với ngăn xếp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `'-12'`; (b) `-12`; (c) `False`; (d) `True`. Bài 3: đầu hàm xử lý chuỗi là một số.
- **Mức 2 (vừa):** Bài 2: `[` mở A; `1` vào A; `[` mở B; `]` đóng B gắn A (B rỗng); `[` mở C; `2` vào C; `[` mở D; `3` vào D; `]` đóng D gắn C; `]` đóng C gắn A; `]` đóng A ⇒ kết quả `[1,[],[2,[3]]]`. Bài 4: nhớ dùng `nonlocal i`.
- **Mức 3 (sâu):** Bài 5: `max_depth`: biến `d` tăng khi `[`, giảm khi `]`, ghi `best = max(best, d)`. `flatten`: ngăn xếp các `iter(getList())`; gặp danh sách con thì đẩy iterator của nó; gặp số thì ghi ra.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
s1 = "[-12,[3]]"
assert s1[1:4] == "-12" and int(s1[1:4]) == -12
assert "-".isdigit() is False and NestedInteger(0).isInteger() is True

# Bài 2
assert ni_to_str(Solution().deserialize("[1,[],[2,[3]]]")) == "[1,[],[2,[3]]]"


# Bài 3
def parse_stack_todo_ans(s):
    if s[0] != "[":
        return NestedInteger(int(s))
    stack, ans = [], None
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c == "[":
            stack.append(NestedInteger()); i += 1
        elif c == "]":
            done = stack.pop()
            if stack:
                stack[-1].add(done)
            else:
                ans = done
            i += 1
        elif c == ",":
            i += 1
        else:
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            stack[-1].add(NestedInteger(int(s[i:j])))
            i = j
    return ans


rng5 = random.Random(5)
for _ in range(100):
    obj = gen_py(rng5, rng5.randint(0, 5), 4)
    assert ni_to_py(parse_stack_todo_ans(to_str(obj))) == obj


# Bài 4
class SolutionRecursive2:
    def deserialize(self, s):
        i = 0
        n = len(s)

        def parse():
            nonlocal i
            if s[i] == "[":
                i += 1
                ni = NestedInteger()
                while s[i] != "]":
                    if s[i] == ",":
                        i += 1
                    else:
                        ni.add(parse())
                i += 1
                return ni
            j = i + 1
            while j < n and s[j].isdigit():
                j += 1
            ni = NestedInteger(int(s[i:j]))
            i = j
            return ni

        return parse()


rng6 = random.Random(6)
for _ in range(100):
    obj = gen_py(rng6, rng6.randint(0, 5), 4)
    s = to_str(obj)
    assert ni_to_py(SolutionRecursive2().deserialize(s)) == json.loads(s) == obj


# Bài 5
def max_depth(s):
    d = best = 0
    for c in s:
        if c == "[":
            d += 1
            best = max(best, d)
        elif c == "]":
            d -= 1
    return best


def flatten(s):
    root = Solution().deserialize(s)
    if root.isInteger():
        return [root.getInteger()]
    out, stack = [], [iter(root.getList())]
    while stack:
        try:
            child = next(stack[-1])
        except StopIteration:
            stack.pop()
            continue
        if child.isInteger():
            out.append(child.getInteger())
        else:
            stack.append(iter(child.getList()))
    return out


assert max_depth("324") == 0 and max_depth("[]") == 1 and max_depth("[1,[2,[3]]]") == 3
assert max_depth("[" * 24000 + "]" * 24000) == 24000
assert flatten("[1,[2,[3]],4]") == [1, 2, 3, 4]
assert flatten("-7") == [-7] and flatten("[[],[[]]]") == []
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ngăn xếp các danh sách đang mở)

```python
class Solution:
    def deserialize(self, s: str) -> NestedInteger:
        if s[0] != "[":
            return NestedInteger(int(s))
        stack = []
        ans = None
        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c == "[":
                stack.append(NestedInteger())
                i += 1
            elif c == "]":
                done = stack.pop()
                if stack:
                    stack[-1].add(done)
                else:
                    ans = done
                i += 1
            elif c == ",":
                i += 1
            else:
                j = i + 1
                while j < n and s[j].isdigit():
                    j += 1
                stack[-1].add(NestedInteger(int(s[i:j])))
                i = j
        return ans
```

**Checklist**

- [ ] Xử lý riêng trường hợp `s` là **một số** (không bắt đầu bằng `[`)?
- [ ] `[` đẩy `NestedInteger()`; `]` **pop rồi gắn vào cha** (hoặc ghi `ans` nếu hết cha)?
- [ ] Đọc **trọn cụm số** (kể cả dấu `-` và nhiều chữ số), không từng ký tự?
- [ ] Phân biệt `NestedInteger()` (danh sách rỗng) và `NestedInteger(0)` (số 0)?
- [ ] Không dùng đệ quy khi độ sâu có thể ~`25 000`?
- [ ] Test: `"[]"`, `"[[]]"`, số âm, `0`, lồng sâu, chuỗi rộng, đối chiếu `json.loads`?